# ABC2Vec — Notebook 06: Evaluation Benchmark Creation

Creates the first public benchmark for folk tune family retrieval — a novel contribution.

**Benchmarks:**
1. **Same-Tune-Family Retrieval** — from The Session's variant "settings" labels
2. **Tune Type Classification** — ground truth: jig, reel, hornpipe, polka, slip jig, slide
3. **Mode/Key labels** — for clustering evaluation

This benchmark is released as a public artifact alongside the paper.


In [14]:
import os, sys, json, re, collections, hashlib, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
BENCHMARK_DIR = PROJECT_DIR / 'data' / 'benchmark'
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

## 6.1 Scrape The Session Tune Variant Data

The Session (thesession.org) has a public JSON API. Each tune has multiple "settings" (notation variants).
These are our ground-truth tune family labels.

API: `https://thesession.org/tunes/{id}?format=json`

In [15]:
# ─── Scrape The Session API for tune families ───
# The Session has ~40K+ tunes with multiple settings each

SESSION_CACHE = BENCHMARK_DIR / 'session_tunes_cache.json'

def fetch_session_tune(tune_id: int, session=None) -> dict | None:
    """Fetch a tune from The Session's JSON API."""
    url = f"https://thesession.org/tunes/{tune_id}?format=json"
    try:
        s = session or requests
        resp = s.get(url, timeout=15)
        if resp.status_code == 200:
            return resp.json()
        return None
    except Exception:
        return None

def scrape_session_tunes(max_tunes: int = 5000, min_settings: int = 3) -> list[dict]:
    """
    Scrape tunes from The Session that have multiple settings (variants).
    Only keeps tunes with >= min_settings different notations.
    """
    # Check cache first
    if SESSION_CACHE.exists():
        with open(SESSION_CACHE) as f:
            cached = json.load(f)
        print(f"Loaded {len(cached)} tunes from cache")
        return cached
    
    session = requests.Session()
    session.headers.update({'User-Agent': 'ABC2Vec-Research/1.0 (academic research)'})
    
    tune_families = []
    failed = 0
    
    # The Session tune IDs go from 1 to ~40000+
    # Sample a range to get diverse tunes
    tune_ids = list(range(1, min(40001, max_tunes * 4)))
    np.random.seed(42)
    np.random.shuffle(tune_ids)
    
    pbar = tqdm(tune_ids[:max_tunes * 2], desc="Scraping The Session")
    
    for tune_id in pbar:
        if len(tune_families) >= max_tunes:
            break
        
        data = fetch_session_tune(tune_id, session)
        if data is None:
            failed += 1
            continue
        
        settings = data.get('settings', [])
        if len(settings) < min_settings:
            continue
        
        # Extract tune family info
        family = {
            'tune_id': data.get('id', tune_id),
            'name': data.get('name', ''),
            'type': data.get('type', ''),
            'num_settings': len(settings),
            'settings': []
        }
        
        for s in settings:
            setting_info = {
                'setting_id': s.get('id', ''),
                'key': s.get('key', ''),
                'abc': s.get('abc', ''),
            }
            if setting_info['abc']:  # Only keep settings with ABC notation
                family['settings'].append(setting_info)
        
        if len(family['settings']) >= min_settings:
            tune_families.append(family)
        
        pbar.set_postfix({'found': len(tune_families), 'failed': failed})
        
        # Rate limiting — be respectful to The Session
        time.sleep(0.2)
    
    # Cache results
    with open(SESSION_CACHE, 'w') as f:
        json.dump(tune_families, f)
    
    print(f"\nScraped {len(tune_families)} tune families with {min_settings}+ settings")
    return tune_families

# Scrape (uses cache if available)
tune_families = scrape_session_tunes(max_tunes=3000, min_settings=3)

if tune_families:
    total_settings = sum(f['num_settings'] for f in tune_families)
    print(f"\nBenchmark statistics:")
    print(f"  Tune families: {len(tune_families)}")
    print(f"  Total settings: {total_settings}")
    print(f"  Avg settings per family: {total_settings/len(tune_families):.1f}")
    print(f"\nTune type distribution:")
    type_counts = collections.Counter(f['type'] for f in tune_families)
    for t, c in type_counts.most_common(10):
        print(f"    {t}: {c}")

Loaded 2349 tunes from cache

Benchmark statistics:
  Tune families: 2349
  Total settings: 16450
  Avg settings per family: 7.0

Tune type distribution:
    reel: 876
    jig: 561
    hornpipe: 205
    polka: 173
    waltz: 137
    slip jig: 92
    barndance: 88
    march: 68
    strathspey: 61
    slide: 50


## 6.2 Build Same-Tune-Family Retrieval Benchmark

In [16]:
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier

def normalize_session_abc(abc_text: str) -> str:
    """Normalize ABC from The Session for consistent processing."""
    # The Session ABC uses \r\n, strip extra whitespace
    abc_text = abc_text.replace('\r\n', '\n').replace('\r', '\n').strip()
    # Normalize whitespace within lines
    lines = [l.strip() for l in abc_text.split('\n') if l.strip()]
    return ' '.join(lines)

# Build retrieval benchmark
benchmark_entries = []
family_id = 0

for family in tune_families:
    for setting in family['settings']:
        abc_norm = normalize_session_abc(setting['abc'])
        if len(abc_norm) < 20:  # Skip too-short entries
            continue
        
        benchmark_entries.append({
            'family_id': family_id,
            'tune_name': family['name'],
            'tune_type': family['type'].lower(),
            'setting_id': setting['setting_id'],
            'key': setting['key'],
            'abc_body': abc_norm,
        })
    family_id += 1

benchmark_df = pd.DataFrame(benchmark_entries)
print(f"Retrieval benchmark:")
print(f"  Total entries: {len(benchmark_df):,}")
print(f"  Families: {benchmark_df['family_id'].nunique()}")
print(f"  Avg entries per family: {len(benchmark_df) / benchmark_df['family_id'].nunique():.1f}")

# Split into query set and corpus
# For each family, pick one setting as query, rest as corpus
queries = []
corpus = []

for fid, group in benchmark_df.groupby('family_id'):
    entries = group.to_dict('records')
    if len(entries) >= 2:
        # First entry is the query
        queries.append(entries[0])
        # Rest are in the corpus (as positives)
        for e in entries[1:]:
            corpus.append(e)

print(f"\nQuery set: {len(queries)} queries")
print(f"Corpus: {len(corpus)} entries")

# Save
queries_df = pd.DataFrame(queries)
corpus_df = pd.DataFrame(corpus)

queries_df.to_parquet(BENCHMARK_DIR / 'retrieval_queries.parquet', index=False)
corpus_df.to_parquet(BENCHMARK_DIR / 'retrieval_corpus.parquet', index=False)
benchmark_df.to_parquet(BENCHMARK_DIR / 'retrieval_full.parquet', index=False)

print(f"Saved to {BENCHMARK_DIR}")

Retrieval benchmark:
  Total entries: 16,450
  Families: 2349
  Avg entries per family: 7.0

Query set: 2349 queries
Corpus: 14101 entries
Saved to /Volumes/LLModels/ABC2VEC/data/benchmark


## 6.3 Build Tune Type Classification Benchmark

In [17]:
# Use the main IrishMAN test set for classification
test_df = pd.read_parquet(PROCESSED_DIR / 'test.parquet')

# tune_type is empty in the processed data (not preserved during pipeline).
# Infer from meter — reliable for Irish folk: jig=6/8, slip jig=9/8,
# slide=12/8, polka=2/4, reel=4/4 or 2/2, waltz=3/4.
# Note: reels and hornpipes both use 4/4; we default to 'reel' since hornpipes
# are rare in IrishMAN and indistinguishable by meter alone.
METER_TO_TYPE = {
    '6/8':  'jig',
    '9/8':  'slip jig',
    '12/8': 'slide',
    '2/4':  'polka',
    '3/4':  'waltz',
    '4/4':  'reel',
    '2/2':  'reel',
    'C':    'reel',
    'C|':   'reel',
}

if test_df['tune_type'].str.strip().eq('').all():
    print("tune_type column is empty — inferring from meter field")
    test_df = test_df.copy()
    test_df['tune_type'] = test_df['meter'].map(METER_TO_TYPE).fillna('unknown')

# Filter to tunes with known types
VALID_TYPES = ['reel', 'jig', 'hornpipe', 'polka', 'slip jig', 'slide', 'waltz', 'march']
classification_df = test_df[test_df['tune_type'].isin(VALID_TYPES)].copy()

print(f"Classification benchmark:")
print(f"  Total: {len(classification_df)} tunes")
print(f"  Types: {classification_df['tune_type'].nunique()}")
print(f"\n  Distribution:")
for t, c in classification_df['tune_type'].value_counts().items():
    print(f"    {t}: {c}")

# Save
classification_df.to_parquet(BENCHMARK_DIR / 'classification_test.parquet', index=False)
print(f"\nSaved to {BENCHMARK_DIR / 'classification_test.parquet'}")


Classification benchmark:
  Total: 2078 tunes
  Types: 6

  Distribution:
    reel: 970
    jig: 461
    polka: 345
    waltz: 247
    slip jig: 39
    slide: 16

Saved to /Volumes/LLModels/ABC2VEC/data/benchmark/classification_test.parquet


## 6.4 Build Mode/Key Clustering Benchmark

In [18]:
# Clustering benchmark: does the embedding space separate by mode and key?
VALID_MODES = ['major', 'minor', 'dorian', 'mixolydian']

clustering_df = test_df[test_df['mode'].isin(VALID_MODES)].copy()

print(f"Clustering benchmark:")
print(f"  Total: {len(clustering_df)} tunes")
print(f"\n  Mode distribution:")
for m, c in clustering_df['mode'].value_counts().items():
    print(f"    {m}: {c}")

# Save
clustering_df.to_parquet(BENCHMARK_DIR / 'clustering_test.parquet', index=False)
print(f"Saved to {BENCHMARK_DIR / 'clustering_test.parquet'}")

Clustering benchmark:
  Total: 2159 tunes

  Mode distribution:
    major: 1744
    minor: 237
    dorian: 116
    mixolydian: 62
Saved to /Volumes/LLModels/ABC2VEC/data/benchmark/clustering_test.parquet


## 6.5 Benchmark Summary

In [19]:
print("=" * 60)
print("ABC2Vec Evaluation Benchmark — Summary")
print("=" * 60)
print(f"\n1. Same-Tune-Family Retrieval (IrishMAN / The Session):")
print(f"   Queries: {len(queries)}")
print(f"   Corpus:  {len(corpus)}")
print(f"   Metrics: MRR, Recall@1, Recall@5, MAP")
print(f"\n2. Tune Type Classification:")
print(f"   Samples: {len(classification_df)}")
print(f"   Classes: {classification_df['tune_type'].nunique()}")
print(f"   Metrics: Accuracy, F1, confusion matrix")
print(f"\n3. Mode/Key Clustering:")
print(f"   Samples: {len(clustering_df)}")
print(f"   Clusters: {clustering_df['mode'].nunique()} modes")
print(f"   Metrics: Silhouette score, NMI, clustering accuracy")
print(f"\nAll benchmarks saved to: {BENCHMARK_DIR}")
for f in sorted(BENCHMARK_DIR.glob('*.parquet')):
    print(f"  {f.name} ({f.stat().st_size / 1e6:.2f} MB)")


ABC2Vec Evaluation Benchmark — Summary

1. Same-Tune-Family Retrieval (IrishMAN / The Session):
   Queries: 2349
   Corpus:  14101
   Metrics: MRR, Recall@1, Recall@5, MAP

2. Tune Type Classification:
   Samples: 2078
   Classes: 6
   Metrics: Accuracy, F1, confusion matrix

3. Mode/Key Clustering:
   Samples: 2159
   Clusters: 4 modes
   Metrics: Silhouette score, NMI, clustering accuracy

All benchmarks saved to: /Volumes/LLModels/ABC2VEC/data/benchmark
  classification_test.parquet (0.77 MB)
  clustering_test.parquet (0.79 MB)
  retrieval_corpus.parquet (1.68 MB)
  retrieval_full.parquet (1.90 MB)
  retrieval_queries.parquet (0.36 MB)
